# 14 — Subtype classical ML (SVM / RF / KNN)

Same `train_classical_cv` pipeline as notebook 09. Per-fold JSON files saved with the `subtype_` prefix so they don't collide with the original lung-binary results.

In [2]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import yaml

from utils.seed import set_seed
from utils.models import CLASSICAL_REGISTRY
from utils.training import train_classical_cv, save_fold_results
from utils.metrics import aggregate_folds
from utils.stats import format_results_table

with open('../configs/histology_subtype.yaml') as f:
    cfg = yaml.safe_load(f)
set_seed(cfg['seed'])

cache_dir = Path('..') / cfg['data']['cache_dir']
data = np.load(cache_dir / 'features.npz', allow_pickle=True)
X, y = data['X'], data['y']
print('X', X.shape, 'pos rate (aca):', y.mean())

X (2000, 208) pos rate (aca): 0.5


In [3]:
results_dir = Path('..') / cfg['paths']['results']
results_dir.mkdir(parents=True, exist_ok=True)

aggregated = {}
for name, (builder, grid) in CLASSICAL_REGISTRY.items():
    print(f'\n=== {name} ===')
    folds = train_classical_cv(
        X, y, builder, grid,
        n_splits=cfg['cv']['n_splits'],
        inner_splits=cfg['cv']['inner_splits'],
        scoring=cfg['cv']['scoring'],
        seed=cfg['seed'],
        verbose=True,
    )
    save_fold_results(folds, results_dir / f'subtype_{name}.json')
    aggregated[name] = aggregate_folds([f.metrics_calibrated for f in folds])

format_results_table(aggregated)


=== SVM ===
[fold 0] train=1600 test=400
[fold 0] best_params={'C': 10, 'gamma': 0.01, 'kernel': 'rbf'} thr=0.928 AUC=0.999 BalAcc(cal)=0.975
[fold 1] train=1600 test=400
[fold 1] best_params={'C': 10, 'gamma': 0.01, 'kernel': 'rbf'} thr=0.960 AUC=0.997 BalAcc(cal)=0.958
[fold 2] train=1600 test=400
[fold 2] best_params={'C': 100, 'gamma': 0.01, 'kernel': 'rbf'} thr=0.996 AUC=0.999 BalAcc(cal)=0.875
[fold 3] train=1600 test=400
[fold 3] best_params={'C': 10, 'gamma': 0.01, 'kernel': 'rbf'} thr=0.870 AUC=1.000 BalAcc(cal)=0.995
[fold 4] train=1600 test=400
[fold 4] best_params={'C': 100, 'gamma': 0.01, 'kernel': 'rbf'} thr=0.996 AUC=0.999 BalAcc(cal)=0.877

=== RF ===
[fold 0] train=1600 test=400
[fold 0] best_params={'max_depth': 10, 'min_samples_split': 2, 'n_estimators': 500} thr=0.688 AUC=0.998 BalAcc(cal)=0.965
[fold 1] train=1600 test=400
[fold 1] best_params={'max_depth': None, 'min_samples_split': 2, 'n_estimators': 500} thr=0.722 AUC=0.994 BalAcc(cal)=0.962
[fold 2] train=1600

c:\Users\User\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] Не удается найти указанный файл
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\User\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
        "wmic CPU Get NumberOfCores /Format:csv".split(),
        capture_output=True,
        text=True,
    )
  File "c:\Users\User\anaconda3\Lib\subprocess.py", line 554, in run
    with Popen(*popenargs, **kwargs) as process:
         ~~~~~^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\User\anaconda3\Lib\subprocess.py", line 1039, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

[fold 0] best_params={'n_neighbors': 7, 'p': 1, 'weights': 'distance'} thr=1.000 AUC=0.998 BalAcc(cal)=0.865
[fold 1] train=1600 test=400
[fold 1] best_params={'n_neighbors': 5, 'p': 2, 'weights': 'distance'} thr=1.000 AUC=0.993 BalAcc(cal)=0.930
[fold 2] train=1600 test=400
[fold 2] best_params={'n_neighbors': 7, 'p': 2, 'weights': 'distance'} thr=1.000 AUC=0.998 BalAcc(cal)=0.880
[fold 3] train=1600 test=400
[fold 3] best_params={'n_neighbors': 5, 'p': 2, 'weights': 'distance'} thr=1.000 AUC=0.996 BalAcc(cal)=0.932
[fold 4] train=1600 test=400
[fold 4] best_params={'n_neighbors': 5, 'p': 1, 'weights': 'distance'} thr=1.000 AUC=0.996 BalAcc(cal)=0.903


,accuracy,sensitivity,specificity,precision,f1,balanced_accuracy,auc_roc,pr_auc
model,,,,,,,,
SVM,0.936 ± 0.056,0.873 ± 0.113,0.999 ± 0.002,0.999 ± 0.002,0.929 ± 0.065,0.936 ± 0.056,0.999 ± 0.001,0.999 ± 0.002
RF,0.966 ± 0.014,0.939 ± 0.022,0.992 ± 0.009,0.992 ± 0.010,0.964 ± 0.015,0.965 ± 0.014,0.998 ± 0.002,0.997 ± 0.005
KNN,0.902 ± 0.030,0.805 ± 0.061,0.999 ± 0.002,0.999 ± 0.003,0.890 ± 0.037,0.902 ± 0.030,0.996 ± 0.002,0.996 ± 0.003
